# Field Scan : détecteur de centres de bananiers (carte de chaleur, 5 cm par pixel)

**Pourquoi** : le détecteur à boîtes (DeepForest, lecture à 8-10 cm) plafonne en plantation dense (rappel ≤ 0,5 même à seuil bas) et rate les jeunes plants (10 à 15 pixels). La métrique de l'orga ne porte que sur les **centres** (appariement à 1 m).

**Ce modèle** : ResNet-18 pré-entraîné ImageNet + décodeur à fusion d'étages (type FPN) ; sortie : carte de probabilité des centres au quart de la résolution ; un pied = un maximum local (fenêtre de ± 0,6 m) au-dessus d'un seuil. Pas de boîtes ni de suppression des doublons : deux couronnes jointives donnent deux pics. Perte focale de CenterNet (une gaussienne par pied, écart-type proportionnel à la couronne). Lecture à **5 cm par pixel** (couronne adulte ≈ 60 px).

**Données** (jeu Kaggle `field-scan-comptage`, nouvelle version) : `deepforest_5cm` (ReforesTree, entraînement Flora Pluas, Nestor Macias, Leonor Aspiazu), `roboflow_centres_5cm` (Plants-bana et Count banana plants train ; validation et test de Count banana plants), `validation_points_5cm` (référence manuelle MM1-MM2). Mêmes découpages que pour DeepForest.

**Règle de choix fixée avant les résultats** : à chaque époque et pour chaque seuil, F1 (appariement à 1 m = 20 px) sur la référence manuelle MM1-MM2 en **mères seules** (rejets et incertains ignorés) et sur la validation dense ; on retient le couple (époque, seuil) qui maximise la **moyenne**. Le test dense n'est regardé qu'une fois, à la fin. Comparaison au modèle DeepForest retenu (v2, lecture à 8 cm, seuil 0,5) : MM mères seules 0,712, validation dense 0,626 (mesures locales).

**Avant de lancer** : Input = `field-scan-comptage` (dernière version) ; GPU ; Internet ; Save & Run All. Durée : environ 1 h ; une ligne par époque.

## 1. Données

In [ ]:
import os, glob, zipfile, shutil, json, time
# Sources (zip ou dossier décompressé par Kaggle), reconnues à un fichier témoin et à leur nom
def localiser(motif_zip, temoin, mot):
    z = [p for p in glob.glob(f'/kaggle/input/**/{motif_zip}', recursive=True) if os.path.isfile(p)]
    if z:
        return 'zip', z[0]
    d = [os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/{temoin}', recursive=True) if mot in p]
    assert d, f'Introuvable : {motif_zip} (ou un dossier contenant {temoin} et « {mot} »)'
    return 'dossier', d[0]

SOURCES = {'rt': localiser('deepforest_5cm*.zip', 'val.csv', '5cm'),
           'rf': localiser('roboflow_centres_5cm*.zip', 'val_dense.csv', '5cm'),
           'mm': localiser('validation_points_5cm*.zip', 'points.csv', 'validation_points')}
print(SOURCES)
D = '/tmp/d'
SORTIE = '/kaggle/working/modele_centres'
shutil.rmtree(D, ignore_errors=True); os.makedirs(f'{D}/images'); os.makedirs(SORTIE, exist_ok=True)
for sous, (genre, chemin) in SOURCES.items():
    os.makedirs(f'{D}/{sous}')
    if genre == 'zip':
        with zipfile.ZipFile(chemin) as z:
            for m in z.infolist():
                c_ = m.filename.replace('\\', '/')
                if c_.endswith('/'):
                    continue
                cible = f'{D}/images/{os.path.basename(c_)}' if '/images/' in '/' + c_ else f'{D}/{sous}/{os.path.basename(c_)}'
                with z.open(m) as src, open(cible, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    else:
        for f in glob.glob(f'{chemin}/*.csv') + glob.glob(f'{chemin}/*.json'):
            shutil.copy(f, f'{D}/{sous}/')
        for f in glob.glob(f'{chemin}/**/images/*', recursive=True):
            os.symlink(f, f'{D}/images/{os.path.basename(f)}')
print(len(os.listdir(f'{D}/images')), 'images ;', {s: sorted(os.listdir(f'{D}/{s}')) for s in SOURCES})


## 2. Versions

In [ ]:
import torch, torchvision, platform, numpy as np, pandas as pd
VERSIONS = {'torch': torch.__version__, 'torchvision': torchvision.__version__, 'python': platform.python_version(),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'}
print(VERSIONS)
assert torch.cuda.is_available(), 'Pas de GPU : Settings > Accelerator > GPU'
DEV = 'cuda'


## 3. Échantillons

In [ ]:
# Annotations -> centres (pixels) et taille de couronne (moyenne des côtés de la boîte)
def centres(csv):
    d = pd.read_csv(csv)
    d = d[(d.xmax > d.xmin) & (d.ymax > d.ymin)]
    d = d.assign(cx=(d.xmin + d.xmax) / 2, cy=(d.ymin + d.ymax) / 2, taille=((d.xmax - d.xmin) + (d.ymax - d.ymin)) / 2)
    return {k: g[['cx', 'cy', 'taille']].to_numpy('float32') for k, g in d.groupby('image_path')}

TRAIN_RT, TRAIN_RF = centres(f'{D}/rt/train.csv'), centres(f'{D}/rf/train_roboflow.csv')
VAL_DENSE, TEST_DENSE = centres(f'{D}/rf/val_dense.csv'), centres(f'{D}/rf/test_dense.csv')
# Tuiles de validation et de test SANS bananier : à garder (zéro pied à trouver)
for csv, dico in ((f'{D}/rf/val_dense.csv', VAL_DENSE), (f'{D}/rf/test_dense.csv', TEST_DENSE)):
    for nom in pd.read_csv(csv).image_path.unique():
        dico.setdefault(nom, np.zeros((0, 3), 'float32'))
PTS_MM, ZONES_MM = pd.read_csv(f'{D}/mm/points.csv'), pd.read_csv(f'{D}/mm/zones.csv')

# Équilibre (même règle que pour DeepForest) : ReforesTree répété jusqu'à au moins 25 % des tuiles d'entraînement
REPETITIONS = max(1, int(np.ceil(0.25 * len(TRAIN_RF) / (0.75 * len(TRAIN_RT)))))
ECHANTILLONS = [(k, v) for k, v in TRAIN_RT.items()] * REPETITIONS + list(TRAIN_RF.items())
print(f'ReforesTree : {len(TRAIN_RT)} tuiles x {REPETITIONS} ; Roboflow : {len(TRAIN_RF)} tuiles ; '
      f'{len(ECHANTILLONS)} échantillons d\'entraînement')
print(f'Validation dense : {len(VAL_DENSE)} tuiles, {sum(len(v) for v in VAL_DENSE.values())} bananiers ; '
      f'test dense : {len(TEST_DENSE)} tuiles, {sum(len(v) for v in TEST_DENSE.values())} bananiers')
print('Référence manuelle :', PTS_MM.groupby(['zone_id', 'type']).size().to_dict())
tailles = np.concatenate([v[:, 2] for v in TRAIN_RF.values()] + [v[:, 2] for v in TRAIN_RT.values()])
print(f'Taille des couronnes (entraînement) : médiane {np.median(tailles):.0f} px, quartiles {np.percentile(tailles, 25):.0f}-{np.percentile(tailles, 75):.0f}')


## 4. Jeu de données, augmentations et cartes cibles

In [ ]:
import random, math
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader

CROP, STRIDE = 448, 4                       # entraînement sur des découpes de 448 px ; carte de sortie au quart
MOY = np.array([0.485, 0.456, 0.406], 'float32'); ECT = np.array([0.229, 0.224, 0.225], 'float32')

def carte_cible(pts, h, w):
    """Une gaussienne par pied, centrée sur la cellule du centre (valeur 1 au pic) ; écart-type = taille / 24."""
    H, W = (h + STRIDE - 1) // STRIDE, (w + STRIDE - 1) // STRIDE
    hm = np.zeros((H, W), 'float32')
    for cx, cy, t in pts:
        i, j = int(cx // STRIDE), int(cy // STRIDE)
        if not (0 <= i < W and 0 <= j < H):
            continue
        s = max(0.8, t / STRIDE / 6)
        r = int(3 * s) + 1
        x0, x1, y0, y1 = max(0, i - r), min(W, i + r + 1), max(0, j - r), min(H, j + r + 1)
        xx, yy = np.arange(x0, x1)[None, :], np.arange(y0, y1)[:, None]
        hm[y0:y1, x0:x1] = np.maximum(hm[y0:y1, x0:x1], np.exp(-((xx - i) ** 2 + (yy - j) ** 2) / (2 * s * s)))
    return hm

def tourner(arr, pts, k):
    """Rotation de k quarts de tour (sens direct, comme np.rot90) de l'image et des points."""
    for _ in range(k):
        n = arr.shape[1]
        arr = np.rot90(arr)
        pts = np.c_[pts[:, 1], n - pts[:, 0], pts[:, 2]] if len(pts) else pts
    return np.ascontiguousarray(arr), pts

class Tuiles(Dataset):
    def __init__(self, echantillons):
        self.e = echantillons
    def __len__(self):
        return len(self.e)
    def __getitem__(self, i):
        nom, pts = self.e[i]
        im = Image.open(f'{D}/images/{nom}').convert('RGB')
        pts = pts.copy()
        e = random.uniform(0.8, 1.25)                                   # variation d'échelle
        im = im.resize((max(1, round(im.width * e)), max(1, round(im.height * e))), Image.BILINEAR)
        pts[:, :3] *= e
        for f in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
            im = f(im).enhance(random.uniform(0.8, 1.2))
        a = np.asarray(im)
        x0, y0 = random.randint(0, max(0, a.shape[1] - CROP)), random.randint(0, max(0, a.shape[0] - CROP))
        cadre = np.zeros((CROP, CROP, 3), 'uint8')
        morceau = a[y0:y0 + CROP, x0:x0 + CROP]
        cadre[:morceau.shape[0], :morceau.shape[1]] = morceau
        pts[:, 0] -= x0; pts[:, 1] -= y0
        pts = pts[(pts[:, 0] >= 0) & (pts[:, 0] < CROP) & (pts[:, 1] >= 0) & (pts[:, 1] < CROP)]
        if random.random() < 0.5:
            cadre = np.ascontiguousarray(cadre[:, ::-1]); pts[:, 0] = CROP - pts[:, 0]
        cadre, pts = tourner(cadre, pts, random.randint(0, 3))
        x = ((cadre.astype('float32') / 255 - MOY) / ECT).transpose(2, 0, 1)
        return torch.from_numpy(x), torch.from_numpy(carte_cible(pts, CROP, CROP)[None])


## 5. Réseau, perte, pics et mesure

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

class DetecteurCentres(nn.Module):
    """ResNet-18 (ImageNet) + décodeur à fusion d'étages (type FPN) -> carte des centres au quart de la résolution."""
    def __init__(self, c=64):
        super().__init__()
        r = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.souche = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.etages = nn.ModuleList([r.layer1, r.layer2, r.layer3, r.layer4])
        self.lat = nn.ModuleList([nn.Conv2d(k, c, 1) for k in (64, 128, 256, 512)])
        self.lisse = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                                   nn.Conv2d(c, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.tete = nn.Conv2d(c, 1, 1)
        nn.init.constant_(self.tete.bias, -2.19)                       # probabilité initiale ≈ 0,1
    def forward(self, x):
        x = self.souche(x)
        feats = []
        for et in self.etages:
            x = et(x); feats.append(x)
        p = self.lat[3](feats[3])
        for k in (2, 1, 0):
            p = F.interpolate(p, size=feats[k].shape[-2:], mode='nearest') + self.lat[k](feats[k])
        return self.tete(self.lisse(p))                                # logits, au quart de la résolution

def perte_focale(logits, cible):
    """Perte focale des cartes de centres (CenterNet) : pics à 1, pénalité réduite près des pics."""
    p = torch.sigmoid(logits.float()).clamp(1e-4, 1 - 1e-4)
    pos = cible.eq(1).float()
    l_pos = torch.log(p) * (1 - p) ** 2 * pos
    l_neg = torch.log(1 - p) * p ** 2 * (1 - cible) ** 4 * (1 - pos)
    return -(l_pos.sum() + l_neg.sum()) / pos.sum().clamp(min=1)

K_PICS = 7                                                             # un pic = maximum sur ± 3 cellules (± 0,6 m à 5 cm)

def pics(carte, seuil):
    """Maxima locaux de la carte (probabilités) au-dessus du seuil -> centres en pixels de l'image, scores."""
    m = F.max_pool2d(carte[None, None], K_PICS, 1, K_PICS // 2)[0, 0]
    ys, xs = torch.nonzero((carte == m) & (carte >= seuil), as_tuple=True)
    return (np.c_[(xs.cpu().numpy() + 0.5) * STRIDE, (ys.cpu().numpy() + 0.5) * STRIDE],
            carte[ys, xs].float().cpu().numpy())

@torch.no_grad()
def predire_carte(modele, img):
    x = torch.from_numpy(((img.astype('float32') / 255 - MOY) / ECT).transpose(2, 0, 1))[None].to(DEV)
    with torch.autocast('cuda', dtype=torch.float16):
        logits = modele(x)
    return torch.sigmoid(logits.float())[0, 0]

def compter(det, certains, ignores, rayon):
    """Comme evaluer_comptage.evaluer_zone : appariement optimal à rayon ; détections proches d'un point ignoré écartées."""
    n, pris = 0, set()
    if len(det) and len(certains):
        d = cdist(det, certains)
        li, co = linear_sum_assignment(np.where(d <= rayon, d, 1e6))
        ok = d[li, co] <= rayon
        n, pris = int(ok.sum()), set(li[ok].tolist())
    reste = [i for i in range(len(det)) if i not in pris]
    ign = int((cdist(det[reste], ignores).min(1) <= rayon).sum()) if len(ignores) and reste else 0
    return n, len(det) - n - ign, len(certains) - n                    # vrais positifs, faux positifs, oublis

def f1(vp, fp, fn):
    p = vp / (vp + fp) if vp + fp else 0.0; r = vp / (vp + fn) if vp + fn else 0.0
    return round(2 * p * r / (p + r), 4) if p + r else 0.0

SEUILS = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6]
RAYON_PX = 20                                                          # 1 m à 5 cm par pixel

def mesurer(modele, tuiles=None, mm=True):
    """F1 par seuil : tuiles denses (centres de boîtes) et référence manuelle MM (mères seules, rejets ignorés)."""
    modele.eval()
    res = {}
    if tuiles is not None:
        acc = {s: [0, 0, 0] for s in SEUILS}
        for nom, pts in tuiles.items():
            det, sc = pics(predire_carte(modele, np.asarray(Image.open(f'{D}/images/{nom}').convert('RGB'))), min(SEUILS))
            for s in SEUILS:
                for k, v in enumerate(compter(det[sc >= s], pts[:, :2], np.zeros((0, 2)), RAYON_PX)):
                    acc[s][k] += v
        res['dense'] = {s: dict(vp=a[0], fp=a[1], fn=a[2], f1=f1(*a)) for s, a in acc.items()}
    if mm:
        acc = {s: [0, 0, 0] for s in SEUILS}
        for _, z in ZONES_MM.iterrows():
            det, sc = pics(predire_carte(modele, np.asarray(Image.open(f'{D}/images/{z.image_path}').convert('RGB'))), min(SEUILS))
            dedans = (det[:, 0] >= z.x0) & (det[:, 0] <= z.x1) & (det[:, 1] >= z.y0) & (det[:, 1] <= z.y1)
            det, sc = det[dedans], sc[dedans]
            p = PTS_MM[PTS_MM.zone_id == z.zone_id]
            certains = p[p.type == 'mere'][['x', 'y']].to_numpy()
            ignores = p[p.type != 'mere'][['x', 'y']].to_numpy()
            for s in SEUILS:
                for k, v in enumerate(compter(det[sc >= s], certains, ignores, RAYON_PX)):
                    acc[s][k] += v
        res['mm'] = {s: dict(vp=a[0], fp=a[1], fn=a[2], f1=f1(*a)) for s, a in acc.items()}
    return res


## 6. Essai rapide (s'arrête en une minute si quelque chose ne va pas)

In [ ]:
# Essai rapide (moins d'une minute) : un lot, un passage avant/arrière, et l'extraction des pics sur une carte parfaite
jeu = Tuiles(ECHANTILLONS)
x, y = jeu[0]
print('tuile', tuple(x.shape), '| carte', tuple(y.shape), '| pics à 1 :', int((y == 1).sum()))
modele = DetecteurCentres().to(DEV)
lot = torch.stack([jeu[i][0] for i in range(4)]).to(DEV); cib = torch.stack([jeu[i][1] for i in range(4)]).to(DEV)
with torch.autocast('cuda', dtype=torch.float16):
    sortie = modele(lot)
perte = perte_focale(sortie, cib); perte.backward()
print('sortie', tuple(sortie.shape), '| perte', round(float(perte), 3), '| gradients finis :',
      all(torch.isfinite(p.grad).all() for p in modele.parameters() if p.grad is not None))
nom, pts = next(iter(VAL_DENSE.items()))
img = np.asarray(Image.open(f'{D}/images/{nom}').convert('RGB'))
parfaite = torch.from_numpy(carte_cible(pts, img.shape[0], img.shape[1])).to(DEV)
det, _ = pics(parfaite, 0.5)
print('carte parfaite :', compter(det, pts[:, :2], np.zeros((0, 2)), RAYON_PX), 'pour', len(pts), 'bananiers (attendu : tous trouvés, 0 faux)')
del modele, lot, cib, sortie; torch.cuda.empty_cache()


## 7. Entraînement et choix (règle fixée à l'avance)

In [ ]:
REGLAGES = dict(reseau='resnet18 ImageNet + décodeur FPN, sortie au quart', resolution_cm=5, decoupe_px=CROP, stride=STRIDE,
                epoques=30, lot=16, lr_squelette=3e-4, lr_decodeur=1e-3, decroissance_poids=1e-4, programme='OneCycle, 5 % de montée',
                ecretage=5.0, augmentations='échelle 0,8-1,25, luminosité/contraste/saturation 0,8-1,2, miroir, quarts de tour',
                k_pics=K_PICS, rayon_px=RAYON_PX, repetitions_reforestree=int(REPETITIONS), duree_max_h=4.0, graine=2026)
random.seed(2026); np.random.seed(2026); torch.manual_seed(2026)
modele = DetecteurCentres().to(DEV)
squelette = list(modele.souche.parameters()) + list(modele.etages.parameters())
ids = {id(p) for p in squelette}
decodeur = [p for p in modele.parameters() if id(p) not in ids]
opt = torch.optim.AdamW([{'params': squelette, 'lr': REGLAGES['lr_squelette']}, {'params': decodeur, 'lr': REGLAGES['lr_decodeur']}],
                        weight_decay=REGLAGES['decroissance_poids'])
charge = DataLoader(Tuiles(ECHANTILLONS), batch_size=REGLAGES['lot'], shuffle=True, num_workers=4, drop_last=True,
                    pin_memory=True, persistent_workers=True)
prog = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[REGLAGES['lr_squelette'], REGLAGES['lr_decodeur']],
                                           total_steps=REGLAGES['epoques'] * len(charge), pct_start=0.05)
echelle = torch.amp.GradScaler()
HISTO, MEILLEUR, debut = [], None, time.time()
print(f"{len(charge)} lots par époque", flush=True)
for ep in range(1, REGLAGES['epoques'] + 1):
    modele.train(); t0, somme, n = time.time(), 0.0, 0
    for x, y in charge:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = modele(x)
        perte = perte_focale(logits, y)
        if not torch.isfinite(perte):
            raise RuntimeError(f'perte non finie à l\'époque {ep}')
        opt.zero_grad(set_to_none=True)
        echelle.scale(perte).backward(); echelle.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(modele.parameters(), REGLAGES['ecretage'])
        echelle.step(opt); echelle.update(); prog.step()
        somme += float(perte); n += 1
    m = mesurer(modele, VAL_DENSE)
    # Règle fixée à l'avance : couple (époque, seuil) qui maximise la moyenne des F1 (MM mères seules, validation dense)
    lignes = [dict(epoque=ep, seuil=s, f1_mm=m['mm'][s]['f1'], f1_dense=m['dense'][s]['f1'],
                   moyenne=round((m['mm'][s]['f1'] + m['dense'][s]['f1']) / 2, 4)) for s in SEUILS]
    HISTO += lignes
    b = max(lignes, key=lambda r: r['moyenne'])
    if MEILLEUR is None or b['moyenne'] > MEILLEUR['moyenne']:
        MEILLEUR = b; torch.save(modele.state_dict(), f'{SORTIE}/meilleur.pt')
    print(f"époque {ep:2d} : perte {somme / n:.3f} | {(time.time() - t0) / 60:.1f} min | meilleur seuil {b['seuil']} : "
          f"MM {b['f1_mm']:.3f}, dense {b['f1_dense']:.3f}, moyenne {b['moyenne']:.3f} | retenu jusqu'ici : époque "
          f"{MEILLEUR['epoque']} ({MEILLEUR['moyenne']:.3f})", flush=True)
    if (time.time() - debut) / 3600 > REGLAGES['duree_max_h']:
        print('Durée maximale atteinte : arrêt propre', flush=True); break
pd.DataFrame(HISTO).to_csv(f'{SORTIE}/choix.csv', index=False)
print('Retenu :', MEILLEUR)


## 8. Test (une seule fois), export ONNX et vérification

In [ ]:
# Test (une seule fois) : Count banana plants, partie test, au seuil retenu ; puis export ONNX et vérification
modele.load_state_dict(torch.load(f'{SORTIE}/meilleur.pt', map_location=DEV)); modele.eval()
s = MEILLEUR['seuil']
t = mesurer(modele, TEST_DENSE, mm=False)['dense'][s]
p_ = t['vp'] / max(t['vp'] + t['fp'], 1); r_ = t['vp'] / max(t['vp'] + t['fn'], 1)
print(f"Test dense (seuil {s}) : précision {p_:.3f}, rappel {r_:.3f}, F1 {t['f1']:.3f}, "
      f"rapport réel / détecté {(t['vp'] + t['fn']) / max(t['vp'] + t['fp'], 1):.2f}")
print('Rappel des références : DeepForest v2 au seuil 0,5 (lecture à 10 cm) : F1 0,559, rapport 2,26')

!pip -q install onnxruntime
import onnxruntime as ort
cpu = DetecteurCentres(); cpu.load_state_dict(torch.load(f'{SORTIE}/meilleur.pt', map_location='cpu')); cpu.eval()
chemin_onnx = f'{SORTIE}/centres_5cm.onnx'
torch.onnx.export(cpu, torch.randn(1, 3, 512, 512), chemin_onnx, input_names=['image'], output_names=['logits'], opset_version=17,
                  dynamic_axes={'image': {2: 'h', 3: 'l'}, 'logits': {2: 'h4', 3: 'l4'}}, dynamo=False)
img = np.asarray(Image.open(f"{D}/images/{ZONES_MM.image_path.iloc[0]}").convert('RGB'))
xin = ((img.astype('float32') / 255 - MOY) / ECT).transpose(2, 0, 1)[None]
ref = cpu(torch.from_numpy(xin)).detach().numpy()
sess = ort.InferenceSession(chemin_onnx, providers=['CPUExecutionProvider'])
t0 = time.time(); out = sess.run(None, {'image': xin})[0]; duree = time.time() - t0
ha = img.shape[0] * img.shape[1] * 0.05 ** 2 / 1e4
print(f"ONNX : écart maximal {np.abs(out - ref).max():.2e} ; {os.path.getsize(chemin_onnx) / 1e6:.0f} Mo ; "
      f"{duree:.2f} s pour {ha:.2f} ha sur le processeur de Kaggle, soit {duree / ha:.0f} s par hectare")
json.dump(dict(versions=VERSIONS, reglages=REGLAGES, retenu=MEILLEUR, seuil=s, test_dense=t,
               normalisation=dict(moyenne=MOY.tolist(), ecart_type=ECT.tolist()), entree='RGB 0-255 / 255, normalisé, NCHW',
               sortie='logits au quart de la résolution ; sigmoïde puis maxima locaux (fenêtre k_pics) au-dessus du seuil',
               regle='couple (époque, seuil) maximisant la moyenne des F1 MM1-MM2 (mères seules, rejets ignorés) et '
                     'validation dense (Count banana plants, valid) ; test dense regardé une fois'),
          open(f'{SORTIE}/centres_5cm.json', 'w'), indent=2, ensure_ascii=False, default=float)
print(sorted(os.listdir(SORTIE)))


## 9. Ensuite, sur ton ordinateur

Télécharge `centres_5cm.onnx` et `centres_5cm.json` (onglet Output, environ 50 Mo), range-les dans `Field-Scan/modeles/`, puis mesure-le avec `scripts/detecter_centres.py` sur MM1-MM2, sur Carlos Vera Guevara et sur El Oro, comme les modèles DeepForest.